# 12 — Joint B/C rescue, reverse harm, and blinded qualitative review

This notebook completes the quantitative portion of Reviewer 2, Major
Comment 1. A label-event rescue means B and C are both wrong while D is
correct; harm is the reverse. Strict study rescue/harm and dominant
error-count transitions are also reported. Patient/source-clustered
intervals, paired sign-flip tests, and prespecified Holm corrections are
used. The notebook then creates two independently completed, blinded
qualitative-review forms. Quantitative completion does not imply that
the human qualitative review is complete.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
sys.path[:] = [
    entry for entry in sys.path
    if Path(entry or ".").resolve() != implementation_dir
]
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
POST_ROOT = PATHS["root"] / "post_rerun"
POST_ROOT.mkdir(parents=True, exist_ok=True)
print("Code:", RERUN_DIR)
print("Output:", POST_ROOT)

In [ ]:
import importlib
from datetime import datetime, timezone
import numpy as np, pandas as pd
from rerun_code.common import write_json
from rerun_code.config import sha256_path
from rerun_code.report_labeler import LABELS_13
import rerun_code.post_rerun as post_module
post_module = importlib.reload(post_module)
if int(getattr(post_module, "POST_RERUN_API_VERSION", 0)) < 1:
    raise ImportError("Copy the updated rerun_code/post_rerun.py and restart the kernel.")
from rerun_code.post_rerun import (
    JOINT_COUNT_COLUMNS, bootstrap_cluster_counts, build_blinded_qualitative_materials,
    clustered_sign_flip_p, load_statistical_frame, make_joint_inputs,
    pooled_cluster_counts, qualitative_case_sample, summarize_pooled_joint,
    joint_stratum_statistics, verified_per_study_path,
)
from rerun_code.statistics import holm_adjust

OUTPUT = POST_ROOT / "joint_rescue_qualitative"
QUALITATIVE = OUTPUT / "qualitative_review"
OUTPUT.mkdir(parents=True, exist_ok=True)
QUALITATIVE.mkdir(parents=True, exist_ok=True)
per_study_path, upstream_audit = verified_per_study_path(PATHS)
frame = load_statistical_frame(per_study_path)
reps = int(CONFIG["statistics"]["permutation_replicates"])
boot_reps = int(CONFIG["statistics"]["bootstrap_replicates"])
confidence = float(CONFIG["statistics"]["confidence_level"])
seed = int(CONFIG["statistics"]["seed"])

In [ ]:
joint_rows, pooled_parts, case_rows = [], [], []
grouped = list(frame.groupby(["model_key", "bundle", "source_dataset"], dropna=False))
for index, (keys, group) in enumerate(grouped, start=1):
    inputs = make_joint_inputs(group)
    result, row_counts = joint_stratum_statistics(
        inputs, replicates=boot_reps, confidence_level=confidence, seed=seed
    )
    result.update({"model_key": keys[0], "bundle": keys[1], "source_dataset": keys[2]})
    joint_rows.append(result)
    base = inputs["base"][["query_record_id", "patient_key"]].copy()
    base["model_key"], base["bundle"], base["source_dataset"] = keys
    for column_index, name in enumerate(JOINT_COUNT_COLUMNS):
        base[name] = row_counts[:, column_index]
    for label_index, label in enumerate(LABELS_13):
        base[f"rescue__{label}"] = inputs["rescue"][:, label_index].astype(int)
        base[f"harm__{label}"] = inputs["harm"][:, label_index].astype(int)
        base[f"wrongopp__{label}"] = (
            inputs["wrong_b"][:, label_index] & inputs["wrong_c"][:, label_index]
        ).astype(int)
        base[f"correctopp__{label}"] = (
            ~inputs["wrong_b"][:, label_index] & ~inputs["wrong_c"][:, label_index]
        ).astype(int)
    pooled_parts.append(base)

    err_b = np.asarray(inputs["err_b"], dtype=int)
    err_c = np.asarray(inputs["err_c"], dtype=int)
    err_d = np.asarray(inputs["err_d"], dtype=int)
    masks = {
        "strict_rescue": (err_b > 0) & (err_c > 0) & (err_d == 0),
        "strict_harm": (err_b == 0) & (err_c == 0) & (err_d > 0),
        "dominant_rescue": err_d < np.minimum(err_b, err_c),
        "dominant_harm": err_d > np.maximum(err_b, err_c),
    }
    selected = np.logical_or.reduce(list(masks.values()))
    for row_index in np.flatnonzero(selected):
        tags = [name for name, mask in masks.items() if mask[row_index]]
        rescued = [
            label for label_index, label in enumerate(LABELS_13)
            if bool(inputs["rescue"][row_index, label_index])
        ]
        harmed = [
            label for label_index, label in enumerate(LABELS_13)
            if bool(inputs["harm"][row_index, label_index])
        ]
        query_id = str(base.iloc[row_index]["query_record_id"])
        case_rows.append({
            "case_key": "|".join((str(keys[0]), str(keys[1]), str(keys[2]), query_id)),
            "model_key": keys[0], "bundle": keys[1], "source_dataset": keys[2],
            "query_record_id": query_id,
            "transition_tags": ";".join(tags),
            "b_label_error_count": int(err_b[row_index]),
            "c_label_error_count": int(err_c[row_index]),
            "d_label_error_count": int(err_d[row_index]),
            "jointly_rescued_labels": ";".join(rescued),
            "jointly_harmed_labels": ";".join(harmed),
        })
    print(f"Completed stratum {index}/{len(grouped)}: {keys}")

joint = pd.DataFrame(joint_rows)
joint["event_net_p_holm_28_strata"] = holm_adjust(joint["event_net_signflip_p"])
joint["strict_study_net_p_holm_28_strata"] = holm_adjust(joint["strict_study_net_signflip_p"])
pooled_rows = pd.concat(pooled_parts, ignore_index=True)
cases = pd.DataFrame(case_rows).drop_duplicates("case_key").reset_index(drop=True)

In [ ]:
pooled_summaries = [summarize_pooled_joint(
    pooled_rows, "all_models_and_strata", replicates=boot_reps,
    confidence_level=confidence, seed=seed,
)]
for model_key, subset in pooled_rows.groupby("model_key", sort=True):
    pooled_summaries.append(summarize_pooled_joint(
        subset, model_key, replicates=boot_reps,
        confidence_level=confidence, seed=seed,
    ))
pooled = pd.DataFrame(pooled_summaries)
model_mask = pooled["scope"] != "all_models_and_strata"
pooled.loc[model_mask, "event_net_p_holm_7_models"] = holm_adjust(
    pooled.loc[model_mask, "event_net_signflip_p"]
)
pooled.loc[model_mask, "strict_study_net_p_holm_7_models"] = holm_adjust(
    pooled.loc[model_mask, "strict_study_net_signflip_p"]
)

label_rows = []
for label in LABELS_13:
    columns = [f"rescue__{label}", f"harm__{label}", f"wrongopp__{label}", f"correctopp__{label}"]
    values = np.column_stack([
        pooled_rows[columns].to_numpy(dtype=float), np.ones(len(pooled_rows))
    ])
    counts, cluster_ids = pooled_cluster_counts(pooled_rows, values)
    formulas = {
        "rescue_rate_all": (0, 4), "harm_rate_all": (1, 4),
        "rescue_rate_opportunity": (0, 2), "harm_rate_opportunity": (1, 3),
        "net_rate_all": (0, 1, 4),
    }
    estimates = bootstrap_cluster_counts(
        counts, formulas, replicates=boot_reps,
        confidence_level=confidence, seed=seed,
    )
    totals = counts.sum(axis=0)
    row = {
        "label": label, "n_clusters": len(cluster_ids), "n_design_events": int(totals[4]),
        "rescue_n": int(totals[0]), "harm_n": int(totals[1]),
        "joint_wrong_opportunities": int(totals[2]),
        "joint_correct_opportunities": int(totals[3]),
        "net_signflip_p": clustered_sign_flip_p(
            counts[:, 0] - counts[:, 1], replicates=reps, seed=seed
        ),
    }
    for name, (estimate, low, high, valid) in estimates.items():
        row[name] = estimate; row[f"{name}_ci_low"] = low
        row[f"{name}_ci_high"] = high; row[f"{name}_valid_bootstrap"] = valid
    label_rows.append(row)
by_label = pd.DataFrame(label_rows)
by_label["net_p_holm_13_labels"] = holm_adjust(by_label["net_signflip_p"])

stratum_path = OUTPUT / "joint_b_c_wrong_d_correct_by_stratum.csv"
pooled_path = OUTPUT / "joint_b_c_wrong_d_correct_pooled_and_by_model.csv"
label_path = OUTPUT / "joint_b_c_wrong_d_correct_by_label.csv"
cases_path = OUTPUT / "joint_transition_case_manifest.csv"
joint.to_csv(stratum_path, index=False)
pooled.to_csv(pooled_path, index=False)
by_label.to_csv(label_path, index=False)
cases.to_csv(cases_path, index=False)
display(pooled)
display(by_label)

In [ ]:
sample_size = int(os.environ.get("JAMIA_QUALITATIVE_SAMPLE_SIZE", "60"))
selected = qualitative_case_sample(cases, sample_size=sample_size, seed=seed)
selected_path = QUALITATIVE / "qualitative_case_sample.csv"
selected.to_csv(selected_path, index=False)
materials = build_blinded_qualitative_materials(
    per_study_path, selected, QUALITATIVE, seed=seed
)
completed1 = QUALITATIVE / "qualitative_reviewer1_completed.csv"
completed2 = QUALITATIVE / "qualitative_reviewer2_completed.csv"
qualitative_forms_present = completed1.exists() and completed2.exists()
status = {
    "quantitative_ready": True,
    "qualitative_forms_present": qualitative_forms_present,
    "qualitative_review_complete": False,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    **upstream_audit,
    "n_study_model_stratum_comparisons": int(pooled_rows["n_records"].sum()),
    "n_joint_case_manifest_rows": len(cases),
    "n_blinded_qualitative_cases": len(selected),
    "bootstrap_replicates": boot_reps,
    "permutation_replicates": reps,
    "confidence_level": confidence,
    "seed": seed,
    "quantitative_outputs": {
        str(path): sha256_path(path)
        for path in (stratum_path, pooled_path, label_path, cases_path, selected_path)
    },
    "qualitative_materials": materials,
    "next_action": (
        "Validate and summarize the two completed review forms in the next cell."
        if qualitative_forms_present else
        "Give the two templates to independent reviewers without the analyst-only crosswalk."
    ),
}
write_json(OUTPUT / "notebook12_status.json", status)
print(json.dumps(status, indent=2))

In [ ]:
# Optional second-stage validation and adjudication. Rerun this cell
# after both reviewers return independently completed files. If any
# binary ratings differ, the cell creates an adjudication template and
# remains pending until a third reviewer completes it.
from sklearn.metrics import cohen_kappa_score
completed1 = QUALITATIVE / "qualitative_reviewer1_completed.csv"
completed2 = QUALITATIVE / "qualitative_reviewer2_completed.csv"
if not (completed1.exists() and completed2.exists()):
    print("QUALITATIVE REVIEW PENDING")
    print("Reviewer 1 form:", materials["reviewer1_template"])
    print("Reviewer 2 form:", materials["reviewer2_template"])
else:
    r1 = pd.read_csv(completed1, dtype=str).fillna("")
    r2 = pd.read_csv(completed2, dtype=str).fillna("")
    keys = ["blinded_case_id", "candidate_code"]
    if r1.duplicated(keys).any() or r2.duplicated(keys).any():
        raise AssertionError("Duplicate blinded case/candidate identifiers in a completed form")
    merged = r1.merge(r2, on=keys, suffixes=("_r1", "_r2"), validate="one_to_one")
    if len(merged) != len(r1) or len(merged) != len(r2):
        raise AssertionError("The two reviewers did not assess identical blinded candidates")
    rating_fields = [
        "fabrication_present_0_or_1", "omission_present_0_or_1",
        "clinically_important_error_0_or_1", "overall_acceptable_0_or_1",
    ]
    agreement_rows = []
    disagreement = np.zeros(len(merged), dtype=bool)
    for field in rating_fields:
        a = pd.to_numeric(merged[f"{field}_r1"], errors="coerce")
        b = pd.to_numeric(merged[f"{field}_r2"], errors="coerce")
        if a.isna().any() or b.isna().any() or not a.isin([0, 1]).all() or not b.isin([0, 1]).all():
            raise ValueError(f"Both reviewers must fill {field} with 0 or 1")
        disagreement |= a.ne(b).to_numpy()
        agreement_rows.append({
            "rating": field,
            "n": len(a),
            "percent_agreement": float(a.eq(b).mean()),
            "cohen_kappa": float(cohen_kappa_score(a, b)),
        })
    for suffix in ("r1", "r2"):
        reviewer_ids = merged[f"reviewer_id_{suffix}"].str.strip().unique()
        if len(reviewer_ids) != 1 or not reviewer_ids[0]:
            raise ValueError(f"Reviewer {suffix} must use one nonempty reviewer_id")
    if merged["reviewer_id_r1"].iloc[0] == merged["reviewer_id_r2"].iloc[0]:
        raise ValueError("The two qualitative reviewers must have distinct reviewer IDs")
    agreement = pd.DataFrame(agreement_rows)
    agreement_path = QUALITATIVE / "qualitative_reader_agreement.csv"
    agreement.to_csv(agreement_path, index=False)
    disagreements = merged.loc[disagreement].copy()
    disagreements_path = QUALITATIVE / "qualitative_disagreements.csv"
    disagreements.to_csv(disagreements_path, index=False)
    print("Completed qualitative ratings:", len(merged))
    print("Rows with any binary-rating disagreement:", len(disagreements))
    display(agreement)

    adjudication_template = QUALITATIVE / "qualitative_adjudication_template.csv"
    adjudication_completed = QUALITATIVE / "qualitative_adjudication_completed.csv"
    final = merged[keys].copy()
    for field in rating_fields:
        a = pd.to_numeric(merged[f"{field}_r1"], errors="raise").astype(int)
        b = pd.to_numeric(merged[f"{field}_r2"], errors="raise").astype(int)
        final[f"final_{field}"] = np.where(a.eq(b), a, np.nan)

    if len(disagreements):
        adjudication = disagreements[keys].copy()
        for descriptive in (
            "source_dataset", "reference_report", "reference_labels", "candidate_report"
        ):
            adjudication[descriptive] = disagreements[f"{descriptive}_r1"]
        for field in rating_fields:
            adjudication[f"reviewer1_{field}"] = disagreements[f"{field}_r1"]
            adjudication[f"reviewer2_{field}"] = disagreements[f"{field}_r2"]
            adjudication[f"adjudicated_{field}"] = ""
        adjudication["adjudicator_id"] = ""
        adjudication["adjudication_notes"] = ""
        adjudication.to_csv(adjudication_template, index=False)

    qualitative_complete = len(disagreements) == 0
    if len(disagreements) and adjudication_completed.exists():
        adj = pd.read_csv(adjudication_completed, dtype=str).fillna("")
        if adj.duplicated(keys).any():
            raise AssertionError("Duplicate identifiers in qualitative adjudication")
        if len(adj) != len(disagreements):
            raise AssertionError("Adjudication must contain every disagreement row exactly once")
        adj = disagreements[keys].merge(adj, on=keys, validate="one_to_one")
        adjudicator_ids = adj["adjudicator_id"].str.strip().unique()
        if len(adjudicator_ids) != 1 or not adjudicator_ids[0]:
            raise ValueError("Use one nonempty adjudicator_id on every adjudication row")
        if adjudicator_ids[0] in {
            merged["reviewer_id_r1"].iloc[0], merged["reviewer_id_r2"].iloc[0]
        }:
            raise ValueError("The adjudicator must differ from both qualitative reviewers")
        for field in rating_fields:
            column = f"adjudicated_{field}"
            values = pd.to_numeric(adj[column], errors="coerce")
            if values.isna().any() or not values.isin([0, 1]).all():
                raise ValueError(f"Fill {column} with 0 or 1")
            mapping = dict(zip(zip(adj["blinded_case_id"], adj["candidate_code"]), values.astype(int)))
            mask = final.set_index(keys).index.isin(mapping)
            final.loc[mask, f"final_{field}"] = [
                mapping[key] for key in final.loc[mask, keys].itertuples(index=False, name=None)
            ]
        qualitative_complete = True

    final_path = QUALITATIVE / "qualitative_final_ratings.csv"
    summary_path = QUALITATIVE / "qualitative_summary_by_condition.csv"
    if qualitative_complete:
        if final.filter(like="final_").isna().any().any():
            raise AssertionError("Final qualitative ratings still contain unresolved disagreements")
        crosswalk = pd.read_csv(materials["analyst_only_crosswalk"])
        final = final.merge(crosswalk, on=keys, validate="one_to_one")
        final.to_csv(final_path, index=False)
        summary = final.groupby("condition")[[f"final_{field}" for field in rating_fields]].mean().reset_index()
        summary.to_csv(summary_path, index=False)
        display(summary)
        print("QUALITATIVE REVIEW COMPLETE")
    else:
        print("QUALITATIVE REVIEW PENDING — complete:", adjudication_template)

    status_path = OUTPUT / "notebook12_status.json"
    refreshed = json.loads(status_path.read_text(encoding="utf-8"))
    refreshed.update({
        "qualitative_forms_present": True,
        "qualitative_review_complete": qualitative_complete,
        "n_qualitative_rating_rows": len(merged),
        "n_rows_requiring_adjudication": len(disagreements),
        "reader_agreement": str(agreement_path),
        "disagreements": str(disagreements_path),
        "adjudication_template": str(adjudication_template) if len(disagreements) else None,
        "final_ratings": str(final_path) if qualitative_complete else None,
        "summary_by_condition": str(summary_path) if qualitative_complete else None,
    })
    write_json(status_path, refreshed)